In [1]:
from feat_engin_helpers import *
import data_loaders

data = data_loaders.DataLoader()
Y, cases, unit_ids = load_uc()
long = to_long(Y, cases, unit_ids)

features = []


# def build_unit_feature(feature, unit_ids, plant_to_res, river=False, upstream=None):
#     cols = {}
#     for uid in unit_ids:
#         plant = uid.rsplit("_", 1)[0]
#         res = plant_to_res[plant]
#         print(plant, res)
#         # if upstream:
#         #     res = with_upstream(res, upstream)
#         if river:
#             res = TO_RIVER[res[0]]
#         cols[uid] = feature[res].sum(axis=1)
#     return pd.DataFrame(cols)

#2. Attach raw features
price = read_timeseries("../sintef_project/data/kernel/Historical_day_ahead_price_2015_2025.csv")["Day-ahead price (EUR/MWh)"]
attach_hourly(long, price, "price")
features.append("price")

inflow = read_timeseries("../sintef_project/data/kernel/Historical_inflow_1958_2025.csv")
inflow = inflow.loc['2015-01-01':]
for res in inflow:
    attach_hourly(long, inflow[res], "inflow_"+res)
    features.append("inflow_"+res)

# storage = read_timeseries("../sintef_project/data/kernel/Historical_volume_2015_2024.csv")
# for res in storage:
#     if res == "Hyljelihyl":
#         continue
#     attach_daily(long, storage[res], "storage_"+res)
#     features.append("storage_"+res)

# value = read_timeseries("../sintef_project/data/kernel/Synthetic_water_value_2015_2024.csv")
# for res in value:
#     attach_daily(long, value[res], "value_"+res)
#     features.append("value_"+res)

# min_vol = read_timeseries("../sintef_project/data/extended/Constraint_min_volume.csv")
# for res in min_vol:
#     attach_hourly(long, min_vol[res], "minVolume_"+res)
#     features.append("minVolume_"+res)

# min_flow = read_timeseries("../sintef_project/data/extended/Constraint_min_flow.csv")
# for res in min_flow:
#     attach_hourly(long, min_flow[res], "minFlow_"+res)
#     features.append("minFlow_"+res)

In [2]:
# 3. New features
print(features,"\n")

# Rolling prices
# price_rolling_day = price.rolling(24, min_periods=1).sum()
# price_rolling_week = price.rolling(24 * 7, min_periods=1).sum()
# attach_hourly(long, price_rolling_day, "rolling_prices_day")
# attach_hourly(long, price_rolling_week, "rolling_prices_week")
# features.extend(["rolling_prices_day", "rolling_prices_week"])

# Lagged price 
# for i in [1, 3, 7]:
#     long["lagged_price_"+str(i)] = price.shift(i)
#     features.append("lagged_price_"+str(i))

# Cumulative inflow
# for res in inflow:
#     cum_inflow_3d = inflow[res].rolling(3, min_periods=1).sum()
#     cum_inflow_week = inflow[res].rolling(7, min_periods=1).sum()
#     cum_inflow_month = inflow[res].rolling(30, min_periods=1).sum()
#     attach_daily(long, cum_inflow_3d, "cum_inflow_3d")
#     attach_daily(long, cum_inflow_week, "cum_inflow_week")
#     attach_daily(long, cum_inflow_month, "cum_inflow_month")
#     features.append(["cum_inflow_3d", "cum_inflow_week", "cum_inflow_month"])

#Day to day changes in inflow
# for res in inflow:
#     long["inflow_change_"+res] = storage[res] - storage[res].diff(1)
#     features.append("inflow_change_"+res)

# Upstream volume and inflow
upstream_dict = {"Bandak": ["Vinjevatn", "Botnedalsvatn", "Byrtevatn"], "Byrtevatn": ["Botnedalsvatn"], 
                 "Vinjevatn": ["Vaamarvatn", "Vatjern", "Hyljelihyl"], "Vatjern": ["Langeidvatn"], "Vaamarvatn": ["Totak"], 
                 "Totak": ["Songavatn", "Bitdalsvatn", "Venemo"], "Venemo": ["Hyljelihyl", "Bordalsvatn"],
                 "Hyljelihyl": ["Bordalsvatn", "Foersvatn", "Kjelavatn"], "Foersvatn": ["Langesae", "Kjelavatn", "Bordalsvatn"], 
                 "Kjelavatn": ["Staavatn"]}
# for res in upstream_dict:
#     total_vol = 0
#     for upstr in upstream_dict[res]:
#         if upstr != "Hyljelihyl":
#             total_vol += long["storage_"+upstr]
        
#     long["upstream_vol_"+res] = total_vol
#     features.append("upstream_vol_"+res)

# for res in upstream_dict:
#     total_vol = 0
#     for upstr in upstream_dict[res]:
#         if upstr != "Hyljelihyl":
#             total_vol += long["inflow_"+upstr]
        
#     long["upstream_infl_"+res] = total_vol
#     features.append("upstream_infl_"+res)

# upstream_dict2 = {"Bandak": ["Vinjevatn"], "Byrtevatn": ["Botnedalsvatn"], 
#                  "Vinjevatn": ["Vaamarvatn", "Vatjern"],
#                  "Totak": ["Songavatn", "Bitdalsvatn"],
#                  "Hyljelihyl": ["Foersvatn"], "Foersvatn": ["Kjelavatn"]}
# for res in upstream_dict2:
#     total_vol = 0
#     for upstr in upstream_dict[res]:
#         if upstr != "Hyljelihyl":
#             total_vol += long["storage_"+upstr]
        
#     long["upstream_vol_generator_"+res] = total_vol
#     features.append("upstream_vol_generator_"+res)

# def getVol(res):
#     vol = 0
#     for upstr in upstream_dict[res]:
#         vol += getVol(upstr)
#     if upstr != "Hyljelihyl":
#         vol += long["storage_"+res]
#     return vol

# for res in upstream_dict:
#     total_vol = 0
#     for upstr in upstream_dict[res]:
#         total_vol += getVol(upstr)
        
#     long["upstream_vol_BAAAAA"+res] = total_vol
#     features.append("upstream_vol_BAAAAA"+res)

# Day to day changes in storage
# for res in storage:
#     long["vol_change_"+res] = storage[res] - storage[res].diff(1)
#     features.append("vol_change_"+res)
    
#Check miss ook voor andere horizonten indien praktisch
# for res in value:
#     attach_case_level(long, cases, value[res], "value_"+res+"_end",   offset_days=7)
#     long["price_minus_value_end_"+res] = long["price"] - long["value_"+res+"_end"]
#     features.append("price_minus_value_end_"+res)
    
# for res in value:
#     attach_case_level(long, cases, value[res], "value_"+res+"_start",   offset_days=7)
#     long["price_rank_in_case_"+res] = long.groupby("case_idx")["price"].rank(pct=True)
#     features.append("price_rank_in_case_"+res)

# for res in min_vol:
#     long["storage_minus_minVol_"+res] = long["storage"+res] - long["minVolume_"+res]
#     features.append("storage_minus_minVol_"+res)

#     long["rel_storage_minus_minVol_"+res] = (long["storage"+res] - long["minVolume_"+res])/long["storage"+res]
#     features.append("rel_storage_minus_minVol_"+res)

# long["day_of_week"] = long["timestamp"].dt.dayofweek
# long["month"] = long["timestamp"].dt.month

# features.append("hour_of_day")
# features.append("day_of_week")
# features.append("month")

# plant_reservoirs = {            # fill in from the topology
#     "Hogga":  ["Kjelavatn", "Langesae"],
#     "Tokke":  ["Totak"],
#     # ...
# }
# unit_inflow = pd.DataFrame({
#     uid: inflow[plant_reservoirs[uid.rsplit("_", 1)[0]]].sum(axis=1)
#     for uid in unit_ids
# })
# attach_unit_specific(long, unit_inflow, "inflow_upstream")
# features.append("inflow_upstream")

#experimenteer met rolling dingen voor veelbelovende features: x - aantal dagen/uren
#cum inflow (3, 7, 30 days)
# day to day changes in inflow
# day to day changes in storage
# total and mean inflow over horizon (x days)
# Rolling Trend / Slope e.g. Inflow_7d_avg - Inflow_30d_avg etc.
# Rolling Volatility / Max Inflow
# Rolling Storage Rate of Change Storage_t - Storage_t-7d etc.
# Rolling Storage Min/Max/Percentile Current Storage relative to 30-day Rolling Max
# Rolling Standard Deviation of Storage
# Weekend, season
# distance to min vol constraint and min flow constraint
# vol in reservoir(s) boven res
# min_flow/inflow
# storage - minvol

#VOLGENDE STAP: SPEARMAN CORR MATRIX

['price', 'inflow_Langesae', 'inflow_Staavatn', 'inflow_Kjelavatn', 'inflow_Bordalsvatn', 'inflow_Foersvatn', 'inflow_Hyljelihyl', 'inflow_Venemo', 'inflow_Bitdalsvatn', 'inflow_Songavatn', 'inflow_Totak', 'inflow_Vaamarvatn', 'inflow_Langeidvatn', 'inflow_Vatjern', 'inflow_Vinjevatn', 'inflow_Botnedalsvatn', 'inflow_Byrtevatn', 'inflow_Bandak', 'inflow_r_Vest_Vassdraget'] 



In [3]:
## TO DO!!!!!!!!!!!! kies trainingsset om horizon aan te passen
sub = long[long["case_idx"] % 7 == 0]
train = sub[sub["year"] <= 2021]  

for unit, g in train.groupby("unit", observed=True):
    scores = filter_scores(g, features, "y")
print(scores)

                     feature  correlation       auc  strength  \
9           inflow_Songavatn     0.143653  0.582964  0.082964   
8         inflow_Bitdalsvatn     0.143634  0.582953  0.082953   
1            inflow_Langesae     0.143626  0.582947  0.082947   
2            inflow_Staavatn     0.104255  0.560210  0.060210   
3           inflow_Kjelavatn     0.103954  0.560037  0.060037   
14          inflow_Vinjevatn     0.083565  0.548261  0.048261   
10              inflow_Totak     0.083554  0.548255  0.048255   
11         inflow_Vaamarvatn     0.083441  0.548187  0.048187   
7              inflow_Venemo     0.080223  0.546331  0.046331   
6          inflow_Hyljelihyl     0.073913  0.542687  0.042687   
12        inflow_Langeidvatn     0.073886  0.542671  0.042671   
15      inflow_Botnedalsvatn     0.073845  0.542648  0.042648   
18  inflow_r_Vest_Vassdraget     0.073836  0.542642  0.042642   
5           inflow_Foersvatn     0.073832  0.542628  0.042628   
16          inflow_Byrtev